# Practical 09: Describing the Service to a Cluster

**Deployment, Service and probes, written as YAML and checked before they run**

SCSE3040 Machine Learning Operations &middot; Bennett University &middot; Session 2026-27

| | |
|---|---|
| **Lectures this follows** | L22 |
| **Course Outcome** | CO4 |
| **Lab duration** | 120 minutes |
| **Memory needed** | about 200 MB |
| **Extra software** | nothing beyond the course environment |
| **Marks** | 10 |

## Aim

1. State what a Deployment and a Service each do.
2. Write both as YAML manifests for the delivery time container.
3. Establish that a manifest is correct before it reaches a cluster.
4. Add readiness and liveness probes, and state what each is for.

## What this practical is about

P08 produced an image and ran one container from it by hand. That arrangement
has a limit: when the container stops, it stays stopped, and nothing replaces
it.

An **orchestrator** removes the manual step. The cluster is told the desired
state and is left to maintain it. Three objects are sufficient for this
practical.

- A **Pod** is one or more containers scheduled together. It is the smallest
  unit the cluster places on a machine.
- A **Deployment** holds a desired count of identical Pods and replaces any
  that fail. The count is a standing instruction rather than a one time
  request.
- A **Service** gives those Pods one stable address. Pods are replaced and
  their addresses change; the Service address does not.

This practical is written so that a cluster is optional. Every graded task
concerns writing a correct manifest, which can be done on any machine. Where a
cluster is reachable the notebook also runs the real commands.

That restriction is deliberate and matches the syllabus. L22 treats Kubernetes
as concepts rather than as a tool this course operates. State that boundary
plainly if asked in a viva: it is the honest answer and it is the one the
course intends.

## Before you start

P08 should be complete, with an image named `delivery-api:1.0`.

YAML was introduced in P05. The same rule applies here: spaces only, never
tabs.

`kubectl` and a cluster are optional. If Docker Desktop has Kubernetes enabled,
the notebook runs the real commands as well.

Run the cells from top to bottom. If notebook state becomes confusing, restart
the kernel, clear all outputs, and run again from the first cell.

In [ ]:
# Step 0: check the workbench before we start.
# This cell only looks; it changes nothing. Run it and read the last line.

import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['yaml']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['yaml']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

# Walkthrough

Read each step, then run its cell.

## Step 1: determine whether a cluster is available

This step is optional. If `kubectl` is installed the notebook runs real
commands. If it is not, everything else still works, because writing a correct
manifest is the skill being assessed.

In [ ]:
import shutil
import subprocess
from pathlib import Path

import yaml

WORK = Path("work")
WORK.mkdir(exist_ok=True)

KUBECTL_OK = shutil.which("kubectl") is not None
CLUSTER_OK = False

if KUBECTL_OK:
    probe = subprocess.run(
        ["kubectl", "cluster-info"], capture_output=True,
        text=True, timeout=30,
    )
    CLUSTER_OK = probe.returncode == 0
    print("kubectl found. Cluster reachable:", CLUSTER_OK)
else:
    print("kubectl is not installed on this machine.")
    print("That is fine. Every task today is about writing correct")
    print("YAML, and the notebook checks your YAML itself.")

print()
print("KUBECTL_OK =", KUBECTL_OK, "  CLUSTER_OK =", CLUSTER_OK)


def kubectl(*args):
    """Run one kubectl command, or explain why it was skipped."""
    if not CLUSTER_OK:
        print(f"SKIPPED (no cluster): kubectl {' '.join(args)}")
        return None
    print(f"$ kubectl {' '.join(args)}")
    done = subprocess.run(["kubectl", *args], capture_output=True,
                          text=True, timeout=120)
    print((done.stdout + done.stderr).strip()[-2000:])
    print("-" * 60)
    return done

## Step 2: write the Deployment

Read it from top to bottom. Every manifest opens with the same four keys:
`apiVersion`, `kind`, `metadata` and `spec`.

One feature causes most of the confusion. There are **two** `metadata` blocks.
The outer one names the Deployment. The inner one, beneath `template`, names the
Pods the Deployment will create. The `selector` between them states which Pods
this Deployment is responsible for, and it must match the template's labels
exactly. A mismatch is not a syntax error, so nothing reports it.

In [ ]:
deployment_yaml = """apiVersion: apps/v1
kind: Deployment
metadata:
  name: delivery-api
spec:
  replicas: 3
  selector:
    matchLabels:
      app: delivery-api
  template:
    metadata:
      labels:
        app: delivery-api
    spec:
      containers:
        - name: api
          image: delivery-api:1.0
          ports:
            - containerPort: 8000
          resources:
            requests:
              memory: "256Mi"
              cpu: "100m"
            limits:
              memory: "512Mi"
              cpu: "500m"
"""

(WORK / "deployment.yaml").write_text(deployment_yaml,
                                      encoding="utf-8")
print(deployment_yaml)

`replicas: 3` is a standing instruction rather than a request: the cluster
restores that count whenever it is not met. `resources` states how much the
container requires, which lets the scheduler choose a machine with room and
prevents one container from consuming an entire node.

## Step 3: check the YAML before anybody applies it

A manifest is text. The quickest way to find a fault in it is to load it in
Python and inspect the result, which requires no cluster.

The habit matters more than it appears. In P11 a pipeline performs exactly this
check automatically, before any deployment is attempted, which is what makes the
gate meaningful.

In [ ]:
with open(WORK / "deployment.yaml", encoding="utf-8") as fh:
    deployment = yaml.safe_load(fh)

print("kind      :", deployment["kind"])
print("name      :", deployment["metadata"]["name"])
print("replicas  :", deployment["spec"]["replicas"])

selector = deployment["spec"]["selector"]["matchLabels"]
pod_labels = deployment["spec"]["template"]["metadata"]["labels"]

print("selector  :", selector)
print("pod labels:", pod_labels)
print()
print("labels match:", selector == pod_labels,
      "  <- if this is False, nothing will work")

## Step 4: write the Service

The Service is considerably shorter. Two fields carry the meaning.

`selector` must match the Pod labels declared in the Deployment. That match is
the entire mechanism by which a Service finds its Pods.

`port` is the port the Service answers on. `targetPort` is the port inside the
container. They are permitted to differ, and confusing the two is the second
most frequent fault in this practical.

In [ ]:
service_yaml = """apiVersion: v1
kind: Service
metadata:
  name: delivery-api-svc
spec:
  type: ClusterIP
  selector:
    app: delivery-api
  ports:
    - port: 80
      targetPort: 8000
"""

(WORK / "service.yaml").write_text(service_yaml, encoding="utf-8")
print(service_yaml)

`ClusterIP` means reachable from inside the cluster only, and it is the correct
default. `NodePort` and `LoadBalancer` widen that exposure and should be chosen
only when external access is actually required.

## Step 5: confirm the Service can find the Pods

Load both files and compare the two strings that must agree. The check takes
three lines and prevents a failure that produces no error message anywhere: the
Deployment runs, the Service exists, and no request ever reaches a Pod.

In [ ]:
with open(WORK / "service.yaml", encoding="utf-8") as fh:
    service = yaml.safe_load(fh)

svc_selector = service["spec"]["selector"]

print("Service selects :", svc_selector)
print("Pods are labelled:", pod_labels)
print()
if svc_selector == pod_labels:
    print("MATCH. Traffic will reach the pods.")
else:
    print("MISMATCH. The Service would find nothing,")
    print("and Kubernetes would not warn you.")

## Step 6: add the health probes

The cluster must now be told what healthy means. Both probes call the `/health`
endpoint written in P07, which is why that endpoint was kept inexpensive.

Compare the two `initialDelaySeconds` values and consider why they differ.
Readiness begins asking early, because traffic should arrive as soon as the
service can serve it. Liveness waits considerably longer, because terminating a
container that is merely still starting produces a restart loop from which the
service never recovers.

In [ ]:
probes = {
    "readinessProbe": {
        "httpGet": {"path": "/health", "port": 8000},
        "initialDelaySeconds": 5,
        "periodSeconds": 10,
    },
    "livenessProbe": {
        "httpGet": {"path": "/health", "port": 8000},
        "initialDelaySeconds": 30,
        "periodSeconds": 20,
        "failureThreshold": 3,
    },
}

container = deployment["spec"]["template"]["spec"]["containers"][0]
container.update(probes)

(WORK / "deployment.yaml").write_text(
    yaml.safe_dump(deployment, sort_keys=False), encoding="utf-8"
)

print(yaml.safe_dump(container, sort_keys=False))

The manifest was edited as a Python dictionary and written back. That technique
is worth retaining: it is how a pipeline changes an image tag automatically in
P11, without a human editing YAML.

## Step 7: what each probe does when it fails

The distinction below is examinable and is worth stating aloud until it is
automatic.

In [ ]:
print("readiness fails -> the pod is taken OUT of the Service")
print("                   no traffic arrives; the pod keeps running")
print()
print("liveness  fails -> the container is KILLED and restarted")
print()
print("So:")
print("  slow to start        -> raise readiness initialDelaySeconds")
print("  hung, needs a kick   -> that is what liveness is for")
print("  only a liveness probe-> traffic hits a pod that is not ready")

## Step 8: apply the manifests to a cluster, if one is available

These are the real commands. They execute here only if a cluster is reachable.
Otherwise they are printed, which is sufficient for the notes.

`apply` is declarative. Running it twice causes no harm, because it makes the
cluster match the file rather than performing an action.

In [ ]:
kubectl("apply", "-f", str(WORK / "deployment.yaml"))
kubectl("apply", "-f", str(WORK / "service.yaml"))
kubectl("get", "deployment", "delivery-api")
kubectl("get", "pods", "-l", "app=delivery-api")
kubectl("get", "service", "delivery-api-svc")

print()
print("The commands above, for your notes:")
for line in [
    "kubectl apply -f deployment.yaml",
    "kubectl apply -f service.yaml",
    "kubectl get pods -l app=delivery-api",
    "kubectl describe pod <pod-name>      # read Events at the end",
    "kubectl logs <pod-name>",
    "kubectl rollout status deployment/delivery-api",
    "kubectl rollout undo deployment/delivery-api",
    "kubectl delete -f deployment.yaml -f service.yaml",
]:
    print("   ", line)

## Step 9: reading a failing deployment

These four states account for most of what goes wrong. Learning them now saves
considerable time later, because each one points at a different cause.

In [ ]:
states = [
    ("Pending", "no machine has room, or a volume is missing"),
    ("ContainerCreating", "pulling the image; normal for a minute"),
    ("ImagePullBackOff", "wrong image name or tag, or no access"),
    ("CrashLoopBackOff", "it starts, dies, restarts, forever"),
    ("Running", "the process started -- which is not the same as working"),
]

for state, meaning in states:
    print(f"  {state:<20} {meaning}")

print()
print("First command every time: kubectl describe pod <name>")
print("Then read the Events list at the very bottom. The reason")
print("is almost always there.")

# Your turn

The walkthrough is complete. The remaining work is yours.

There are **three tasks**, in increasing order of difficulty, each with a hint.

Where a `# TODO` comment appears, replace that line with your own code. Do not
rename the variable on the left of the `=` sign. The self-check at the end
looks for exactly those names.

When all three have been attempted, run the **self-check** cell. It may be run
as often as required and it never alters the notebook.

## Task T1: write a Service for a second version

Write `work/service-v2.yaml`, holding a Service named `delivery-api-v2-svc`
which

- is of type `NodePort`,
- selects Pods labelled `app: delivery-api-v2`,
- answers on port `80` and forwards to `targetPort` `8000`.

Load it back with `yaml.safe_load` and store the resulting dictionary in
`T1_service`.

> **Hint.** Follow the Service from Step 4 and change four things: the name, the
> type, the selector value and the ports. Write it with `write_text`, then read
> it back with `yaml.safe_load` so the self-check can inspect it.

In [ ]:
service_v2 = """apiVersion: v1
kind: Service
metadata:
  name: TODO
spec:
  type: TODO
  selector:
    app: TODO
  ports:
    - port: 0
      targetPort: 0
"""

(WORK / "service-v2.yaml").write_text(service_v2, encoding="utf-8")

# TODO: load the file back into a dictionary
T1_service = None

print(T1_service)

## Task T2: probes for a model that loads slowly

A new model requires 40 seconds to load. The Deployment in Step 6 carries a
liveness probe with `initialDelaySeconds: 30`, which would terminate the
container during loading, indefinitely.

Build a dictionary `T2_probes` with two keys, `readinessProbe` and
`livenessProbe`, both calling `/health` on port 8000, where

- readiness begins after 10 seconds,
- liveness begins after 60 seconds, comfortably beyond the 40 required.

> **Hint.** Follow the `probes` dictionary from Step 6 and change the two
> `initialDelaySeconds` values. Leave the `httpGet` blocks unchanged.

In [ ]:
# TODO: two probes, both httpGet /health on port 8000,
#       readiness after 10 seconds, liveness after 60.
T2_probes = None

print(T2_probes)

## Task T3: find the mismatch

The two manifests below are valid YAML and would be applied without a single
warning. No traffic would ever reach a Pod.

Load both, identify the fault, and correct it by editing the **Service** so that
it selects the Pods the Deployment actually creates.

Place the corrected Service dictionary in `T3_fixed_service`.

> **Hint.** Print `broken_deploy['spec']['template']['metadata']['labels']` and
> `broken_svc['spec']['selector']` and read them side by side. One word differs.
> Change the Service selector to match the Pod labels exactly.

In [ ]:
broken_deploy = yaml.safe_load("""
apiVersion: apps/v1
kind: Deployment
metadata:
  name: payments-api
spec:
  replicas: 2
  selector:
    matchLabels:
      app: payments-api
  template:
    metadata:
      labels:
        app: payments-api
    spec:
      containers:
        - name: api
          image: payments-api:3.1
""")

broken_svc = yaml.safe_load("""
apiVersion: v1
kind: Service
metadata:
  name: payments-svc
spec:
  selector:
    app: payment-api
  ports:
    - port: 80
      targetPort: 8080
""")

# TODO: fix the Service so its selector matches the pod labels
T3_fixed_service = None

print(T3_fixed_service)

## Self-check

Run the cell below to mark the work.

In [ ]:
# ------------------------------------------------------------------
# SELF-CHECK: run this when you have attempted the tasks above.
# It never breaks your notebook. A task you have not done yet simply
# shows FAIL.
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    """Evaluate one graded condition without ever raising."""
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check('T1 | it is a Service with the right name', lambda: T1_service['kind'] == 'Service' and T1_service['metadata']['name'] == 'delivery-api-v2-svc')
_check('T1 | its type is NodePort and it selects delivery-api-v2', lambda: T1_service['spec']['type'] == 'NodePort' and T1_service['spec']['selector'] == {'app': 'delivery-api-v2'})
_check('T1 | port 80 forwards to targetPort 8000', lambda: T1_service['spec']['ports'][0]['port'] == 80 and T1_service['spec']['ports'][0]['targetPort'] == 8000)
_check('T2 | both probes are present', lambda: set(T2_probes) == {'readinessProbe', 'livenessProbe'})
_check('T2 | both call /health on port 8000', lambda: all(p['httpGet'] == {'path': '/health', 'port': 8000} for p in T2_probes.values()))
_check('T2 | readiness starts at 10s and liveness at 60s', lambda: T2_probes['readinessProbe']['initialDelaySeconds'] == 10 and T2_probes['livenessProbe']['initialDelaySeconds'] == 60)
_check('T2 | liveness waits longer than readiness', lambda: T2_probes['livenessProbe']['initialDelaySeconds'] > T2_probes['readinessProbe']['initialDelaySeconds'])
_check('T3 | the Service selector now matches the pod labels', lambda: T3_fixed_service['spec']['selector'] == broken_deploy['spec']['template']['metadata']['labels'])
_check('T3 | the selector is app: payments-api, spelled correctly', lambda: T3_fixed_service['spec']['selector'] == {'app': 'payments-api'})
_check('T3 | nothing else about the Service was damaged', lambda: T3_fixed_service['kind'] == 'Service' and T3_fixed_service['metadata']['name'] == 'payments-svc' and T3_fixed_service['spec']['ports'][0]['targetPort'] == 8080)

print("==================================================================")
print("SELF-CHECK   Practical 09: Describing the Service to a Cluster")
print("==================================================================")
for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")
print("------------------------------------------------------------------")
_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("==================================================================")
if _passed == len(_results):
    print("Well done. Save the notebook and submit it.")
else:
    print("Read the FAIL lines above, fix those tasks, run this cell again.")

## What to submit

1. This notebook, with every cell run and its output visible.
2. The files `work/deployment.yaml` and `work/service.yaml`.
3. In a markdown cell, two sentences: what happens when a readiness probe
   fails, and what happens when a liveness probe fails.

Name the file `P09_<your-roll-number>.ipynb` before uploading it.

### How this practical is marked

| What is marked | Marks |
|---|---|
| Walkthrough run end to end, manifests written and checked | 3 |
| Task T1, a correct Service manifest | 2 |
| Task T2, probes added with defensible delays | 3 |
| Task T3, the label mismatch found and corrected | 2 |
| **Total** | **10** |

## Read more

- Kubernetes, Deployments: <https://kubernetes.io/docs/concepts/workloads/controllers/deployment/>
- Kubernetes, Services: <https://kubernetes.io/docs/concepts/services-networking/service/>
- Kubernetes, liveness and readiness probes: <https://kubernetes.io/docs/tasks/configure-pod-container/configure-liveness-readiness-startup-probes/>
- Kubernetes, managing resources: <https://kubernetes.io/docs/concepts/configuration/manage-resources-containers/>